In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
# ==============================================================================
#  SETUP AWAL & IMPORTS
# ==============================================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
from google.colab import drive
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

import matplotlib.patches as mpatches

print(f"Versi segmentation-models-pytorch yang digunakan: {smp.__version__}")

# Inisialisasi dataset

In [ ]:
# ==============================================================================
# KONEKSI GOOGLE DRIVE DAN PERSIAPAN PATH
# ==============================================================================
import os # Import os module here
BASE_DATA_DIR = '/content/drive/MyDrive/Draft Skripsi/Dataset'

IMAGE_DIR = os.path.join(BASE_DATA_DIR, 'JPEGImages')
MASK_DIR = os.path.join(BASE_DATA_DIR, 'SegmentationClass')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
#  MEMUAT DAN MEMPROSES DATA (PYTORCH DATASET) - dengan HANDLE KELAS MINORITAS
# ==============================================================================

import os, cv2, numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = (384, 384)
BATCH_SIZE = 4
N_CLASSES = 5   # 0:bg, 1:leaf, 2:bacterial_blight, 3:blast, 4:brown_spot
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==================== COLOR MAPPING (RGB setelah cvtColor) ====================
COLOR2CLASS = {
    (0, 0, 0): 0,
    (255, 0, 0): 1,   # Leaf
    (245, 128, 6): 2, # Blight
    (53, 119, 181): 3,# Blast
    (170, 170, 255): 4# Brown spot
}

# Total unique colors found: 5
# Unique colors (BGR): [(0, 0, 0), (0, 0, 255), (6, 128, 245), (181, 119, 53), (255, 170, 170)]

IDX2NAME = {
    0: "Background",
    1: "Blast",
    2: "Bacterial Blight",
    3: "Leaf",
    4: "Brown Spot"
}

# ==================== Sinkronisasi file ====================
image_files = sorted(os.listdir(IMAGE_DIR))
mask_files = sorted(os.listdir(MASK_DIR))
image_filenames_set = set(os.path.splitext(f)[0] for f in image_files)
mask_filenames_set = set(os.path.splitext(f)[0] for f in mask_files)
common_files = sorted(list(image_filenames_set.intersection(mask_filenames_set)))

image_paths, mask_paths = [], []
for fname in common_files:
    image_path_found, mask_path_found = None, None
    for ext in ['.png', '.jpg', '.jpeg']:
        if os.path.exists(os.path.join(IMAGE_DIR, fname + ext)):
            image_path_found = os.path.join(IMAGE_DIR, fname + ext)
        if os.path.exists(os.path.join(MASK_DIR, fname + ext)):
            mask_path_found = os.path.join(MASK_DIR, fname + ext)
    if image_path_found and mask_path_found:
        image_paths.append(image_path_found)
        mask_paths.append(mask_path_found)

print(f"Jumlah pasangan gambar & mask yang cocok: {len(image_paths)}")

# ==================== Util ====================
def rgb_to_class_mask(mask_rgb, color_map):
    h, w, _ = mask_rgb.shape
    mask_class = np.zeros((h, w), dtype=np.uint8)
    for color, class_idx in color_map.items():
        mask_class[np.all(mask_rgb == color, axis=-1)] = class_idx
    return mask_class

def get_dominant_class(mask_path, color_map):
    mask_rgb = cv2.imread(mask_path)
    mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)
    mask_class = rgb_to_class_mask(mask_rgb, color_map)
    vals, counts = np.unique(mask_class, return_counts=True)
    return vals[np.argmax(counts)]

def has_brown_spot(mask_path):
    mask_rgb = cv2.imread(mask_path)
    mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)
    mask_class = rgb_to_class_mask(mask_rgb, COLOR2CLASS)
    return 4 in np.unique(mask_class)

# ==================== Cari file Brown Spot ====================
brownspot_files = [(img, msk) for img, msk in zip(image_paths, mask_paths) if has_brown_spot(msk)]
print(f"Jumlah file dengan Brown Spot: {len(brownspot_files)}")

# Pastikan minimal 3 file untuk dibagi Train/Val/Test
manual_test   = brownspot_files[:1]   # 1 ke Test
manual_val    = brownspot_files[1:2]  # 1 ke Val
manual_train  = brownspot_files[2:]   # sisanya ke Train

# ==================== Buat dataset tanpa file Brown Spot (untuk split stratify) ====================
image_paths_clean = [img for img, msk in zip(image_paths, mask_paths) if (img, msk) not in brownspot_files]
mask_paths_clean  = [msk for img, msk in zip(image_paths, mask_paths) if (img, msk) not in brownspot_files]

dominant_labels = [get_dominant_class(m, COLOR2CLASS) for m in mask_paths_clean]

# Split train+val vs test
trainval_imgs, test_imgs, trainval_masks, test_masks, y_trainval, y_test = train_test_split(
    image_paths_clean, mask_paths_clean, dominant_labels,
    test_size=0.05, random_state=42, stratify=dominant_labels
)

# Split train vs val
train_imgs, val_imgs, train_masks, val_masks, y_train, y_val = train_test_split(
    trainval_imgs, trainval_masks, y_trainval,
    test_size=0.15/0.95, random_state=42, stratify=y_trainval
)

# Tambahkan file Brown Spot ke masing-masing split
train_imgs  += [img for img, msk in manual_train]
train_masks += [msk for img, msk in manual_train]
val_imgs    += [img for img, msk in manual_val]
val_masks   += [msk for img, msk in manual_val]
test_imgs   += [img for img, msk in manual_test]
test_masks  += [msk for img, msk in manual_test]

print(f"\nTotal gambar yang akan digunakan: {len(image_paths)}")
print(f"Data Latih   : {len(train_imgs)}")
print(f"Data Validasi: {len(val_imgs)}")
print(f"Data Uji     : {len(test_imgs)}")

# ==================== AUGMENTASI ====================
train_transform = A.Compose([
    A.Resize(height=IMG_SIZE[0], width=IMG_SIZE[1]),
    A.Rotate(limit=35, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
    ToTensorV2(),
])

val_test_transform = A.Compose([
    A.Resize(height=IMG_SIZE[0], width=IMG_SIZE[1]),
    A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
    ToTensorV2(),
])

# ==================== Dataset Class ====================
class PaddyDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.color_map = COLOR2CLASS

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)
        mask = rgb_to_class_mask(mask_rgb, self.color_map)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].long()

        return image, mask

# ==================== DataLoader ====================
train_dataset = PaddyDataset(train_imgs, train_masks, transform=train_transform)
val_dataset = PaddyDataset(val_imgs, val_masks, transform=val_test_transform)
test_dataset = PaddyDataset(test_imgs, test_masks, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# # ==================== Cek distribusi label ====================
# def count_labels(dataset, name):
#     counts = np.zeros(N_CLASSES, dtype=np.int64)
#     for _, mask in dataset:
#         mask_np = mask.numpy()
#         unique, c = np.unique(mask_np, return_counts=True)
#         for u, cc in zip(unique, c):
#             counts[u] += cc
#     total = counts.sum()
#     print(f"\nDistribusi label di {name}:")
#     for i, cnt in enumerate(counts):
#         pct = 100.0 * cnt / total if total > 0 else 0
#         print(f"  Class {i} ({IDX2NAME[i]}): {cnt} pixels ({pct:.2f}%)")

# count_labels(train_dataset, "Train")
# count_labels(val_dataset, "Validation")
# count_labels(test_dataset, "Test")


# Train

In [ ]:
# ==============================================================================
#  MEMBANGUN MODEL SEGMENTASI (DeepLabV3+ DENGAN PYTORCH)
# ==============================================================================
# Inisialisasi model DeepLabV3+ dari segmentation-models-pytorch
model = smp.PSPNet(
    encoder_name="resnet34",        # Pilih encoder
    encoder_weights="imagenet",     # Gunakan bobot pre-trained ImageNet
    in_channels=3,                  # Jumlah channel input (RGB)
    classes=N_CLASSES,              # Jumlah kelas output
)

model.to(DEVICE)

print("Model DeepLabV3+ (PyTorch) berhasil dibuat.")

In [ ]:
model

In [ ]:
! pip install torchinfo

In [ ]:
from torchinfo import summary

summary(
    model,
    depth=5,
    input_size=(1, 3, 384, 384)
)

In [ ]:
from torchview import draw_graph

# ukuran input
input_size = (1, 3, 384, 384)

graph = draw_graph(
    model,
    input_size=input_size,
    expand_nested=True,
    device=DEVICE
)

# tampilkan di notebook
graph.visual_graph

# simpan sebagai PNG
# graph.visual_graph.render("pspnet_architecture", format="png")

In [ ]:
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm
import os

# --- Setup parameter ---
EPOCHS = 200
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Loss & optimizer
loss_fn = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# History
train_losses, val_losses = [], []
train_ious, val_ious = [], []

best_val_iou = 0.0
nama_model = "deeplabv3plus"
folder_model_path = os.path.join("/content/drive/MyDrive/Draft Skripsi/Model", nama_model)
os.makedirs(folder_model_path, exist_ok=True)
best_model_path = os.path.join(folder_model_path, f"{nama_model}_best_val_iou_{EPOCHS}.pth")
last_model_path = os.path.join(folder_model_path, f"{nama_model}_last_{EPOCHS}.pth")


# =========================================================
#  FUNGSI TRAINING 1 EPOCH
# =========================================================
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    running_loss, running_iou = 0.0, 0.0

    progress_bar = tqdm(loader, desc="Training", leave=True)
    for x_batch, y_batch in progress_bar:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(x_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)

        # Hitung IoU
        tp, fp, fn, tn = smp.metrics.get_stats(
            preds.argmax(dim=1), y_batch, mode='multiclass', num_classes=N_CLASSES
        )
        iou_score = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro")
        running_iou += iou_score.item() * x_batch.size(0)

        progress_bar.set_postfix(loss=loss.item(), iou=iou_score.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_iou  = running_iou / len(loader.dataset)
    return epoch_loss, epoch_iou


# =========================================================
#  FUNGSI VALIDASI 1 EPOCH
# =========================================================
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss, running_iou = 0.0, 0.0

    progress_bar = tqdm(loader, desc="Validation", leave=True)
    with torch.no_grad():
        for x_batch, y_batch in progress_bar:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            loss = loss_fn(preds, y_batch)

            running_loss += loss.item() * x_batch.size(0)

            tp, fp, fn, tn = smp.metrics.get_stats(
                preds.argmax(dim=1), y_batch, mode='multiclass', num_classes=N_CLASSES
            )
            iou_score = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro")
            running_iou += iou_score.item() * x_batch.size(0)

            progress_bar.set_postfix(val_loss=loss.item(), val_iou=iou_score.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_iou  = running_iou / len(loader.dataset)
    return epoch_loss, epoch_iou


# =========================================================
#  MAIN LOOP
# =========================================================
print("--- Memulai Pelatihan Model ---")
for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    # Training
    train_loss, train_iou = train_one_epoch(model, train_loader, loss_fn, optimizer, DEVICE)
    train_losses.append(train_loss)
    train_ious.append(train_iou)

    # Validation
    val_loss, val_iou = validate_one_epoch(model, val_loader, loss_fn, DEVICE)
    val_losses.append(val_loss)
    val_ious.append(val_iou)

    # Log hasil (selalu di baris baru, tidak ketimpa tqdm)
    print(f"[Epoch {epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train IoU: {train_iou:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}")

    # Simpan best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), best_model_path)
        print(f"  >> Best model updated & saved: {best_model_path}")

# Simpan last model
torch.save(model.state_dict(), last_model_path)
print(f"\n--- Pelatihan selesai ---")
print(f"Last model disimpan ke: {last_model_path}")


In [ ]:
import matplotlib.pyplot as plt
import os

# === Plot Learning Curve ===
plt.figure(figsize=(12,5))

# ---- Loss curve ----
plt.subplot(1,2,1)
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss", marker="o")
plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve - Loss")
plt.legend()
plt.grid(True)

# ---- IoU curve ----
plt.subplot(1,2,2)
plt.plot(range(1, len(train_ious)+1), train_ious, label="Train IoU", marker="o")
plt.plot(range(1, len(val_ious)+1), val_ious, label="Val IoU", marker="s")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("Learning Curve - IoU")
plt.legend()
plt.grid(True)

plt.tight_layout()

# === Simpan ke Google Drive ===
curve_path = os.path.join(folder_model_path, f"{nama_model}_learning_curve_{EPOCHS}.png")
plt.savefig(curve_path, dpi=300)
plt.show()

print(f"Learning curve disimpan ke: {curve_path}")


# Evaluate

In [ ]:
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

# ====== SETUP ======
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_CLASSES = 5   # ganti sesuai dataset
MODEL_PATH = "/content/drive/MyDrive/Draft Skripsi/Model/deeplabv3plus/deeplabv3plus_last_200.pth"

# Inisialisasi DeepLabV3+
model = smp.DeepLabV3Plus(
    encoder_name="resnet34",      # backbone, ganti kalau beda
    encoder_weights=None,         # saat evaluasi tidak perlu pretrain lagi
    in_channels=3,
    classes=N_CLASSES,
).to(DEVICE)

# Load weight hasil training
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

# Loss function
loss_fn = smp.losses.DiceLoss(mode='multiclass')

# ====== EVALUASI ======
test_loss, test_iou = 0.0, 0.0
correct_pixels, total_pixels = 0, 0   # untuk accuracy

with torch.no_grad():
    progress_bar = tqdm(test_loader, desc="[Testing]")
    for x_batch, y_batch in progress_bar:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)

        preds = model(x_batch)
        loss = loss_fn(preds, y_batch)
        test_loss += loss.item() * x_batch.size(0)

        # Pred label
        pred_idx = preds.argmax(dim=1)  # (B,H,W)

        # ====== Pixel-wise Accuracy ======
        correct = (pred_idx == y_batch).sum().item()
        total = y_batch.numel()
        batch_acc = correct / total
        correct_pixels += correct
        total_pixels += total

        # ====== IoU (macro) ======
        tp, fp, fn, tn = smp.metrics.get_stats(
            pred_idx, y_batch, mode="multiclass", num_classes=N_CLASSES
        )
        iou_score = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro")
        test_iou += iou_score.item() * x_batch.size(0)

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "iou": f"{iou_score.item():.4f}",
            "acc": f"{batch_acc:.4f}",
        })

# Rata-rata per dataset
test_loss /= len(test_dataset)
test_iou  /= len(test_dataset)
test_acc   = correct_pixels / total_pixels

print("\n== Evaluation Results ==")
print(f"loss: {test_loss:.4f}")
print(f"iou_score: {test_iou:.4f}")
print(f"accuracy: {test_acc:.4f}")


# Analisis keparahan di data test

In [ ]:
EPOCHS = 200

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from tqdm import tqdm

# ====== Mapping index → nama kelas ======
IDX2NAME = {
    0: "Background",
    1: "Blast",
    2: "Bacterial Blight",
    3: "Leaf",
    4: "Brown Spot",
}

# Balikan COLOR2CLASS dari training (harus sama persis)
CLASS_COLORS = {v: k for k, v in COLOR2CLASS.items()}

# ====== util: decode segmap ======
def decode_segmap(mask, n_classes=len(CLASS_COLORS)):
    h, w = mask.shape
    output = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(n_classes):
        color = CLASS_COLORS.get(i, (0, 0, 0))  # fallback aman
        output[mask == i] = color
    return output

# ====== IoU global ======
def compute_global_iou(tp_list, fp_list, fn_list, num_classes):
    tp = torch.cat(tp_list, dim=0)
    fp = torch.cat(fp_list, dim=0)
    fn = torch.cat(fn_list, dim=0)
    tp_sum = tp.sum(dim=0).float()
    fp_sum = fp.sum(dim=0).float()
    fn_sum = fn.sum(dim=0).float()
    denom = tp_sum + fp_sum + fn_sum
    iou_per_class = torch.where(denom > 0, tp_sum / denom, torch.full_like(denom, float('nan')))
    miou = torch.nanmean(iou_per_class).item()
    return iou_per_class.cpu().numpy(), miou

# ====== Severity Analysis (IRRI SES) ======
def severity_from_pred(pred_idx, gt_idx=None,
                       idx_leaf=3, idx_blight=2, idx_blast=1, idx_brown=4,
                       use_gt_leaf=True):
    pixel_blight = int(np.sum(pred_idx == idx_blight))
    pixel_blast  = int(np.sum(pred_idx == idx_blast))
    pixel_brown  = int(np.sum(pred_idx == idx_brown))
    total_penyakit = pixel_blight + pixel_blast + pixel_brown

    if use_gt_leaf and gt_idx is not None:
        pixel_leaf = int(np.sum(gt_idx == idx_leaf))
    else:
        pixel_leaf = int(np.sum(pred_idx == idx_leaf))

    total_daun = pixel_leaf + total_penyakit
    persen = 0.0 if total_daun <= 0 else 100.0 * total_penyakit / total_daun

    if persen < 1:
        tingkat = "Gejala Awal (SES 1–3)"
    elif persen < 10:
        tingkat = "Ringan (SES 4–5)"
    elif persen < 25:
        tingkat = "Sedang (SES 6)"
    elif persen < 50:
        tingkat = "Parah (SES 7)"
    else:
        tingkat = "Sangat Parah (SES 8–9)"

    penyakit = []
    if pixel_blight > 0: penyakit.append("Bacterial Blight")
    if pixel_blast  > 0: penyakit.append("Blast")
    if pixel_brown  > 0: penyakit.append("Brown Spot")
    if not penyakit:
        tingkat = "Sehat (SES 0)"
        penyakit_str = "Tidak ada penyakit terdeteksi"
    else:
        penyakit_str = ", ".join(penyakit)

    return persen, tingkat, penyakit_str

# ====== Evaluasi + Simpan Hasil ======
def evaluate_and_save(model, test_loader, device, nama_model="DeepLabV3plus", epoch=None):
    model.eval()
    all_tp, all_fp, all_fn = [], [], []
    correct_pixels, total_pixels = 0, 0
    sample_id = 0

    # setup folder hasil: /hasil_analisis/DeepLabV3plus/epoch30/
    if epoch is not None:
        hasil_dir = os.path.join(
            "/content/drive/MyDrive/Draft Skripsi/hasil_analisis",
            str(nama_model),
            f"epoch_{epoch}"
        )
    else:
        hasil_dir = os.path.join(
            "/content/drive/MyDrive/Draft Skripsi/hasil_analisis",
            str(nama_model)
        )
    os.makedirs(hasil_dir, exist_ok=True)

    log_path = os.path.join(hasil_dir, "hasil_evaluasi.txt")
    f_log = open(log_path, "w", encoding="utf-8")

    with torch.no_grad():
        progress = tqdm(test_loader, desc="[Evaluating]", unit="batch")
        for images_b, masks_b in progress:
            images_b, masks_b = images_b.to(device), masks_b.to(device)

            logits_b = model(images_b)
            pred_idx_b = torch.argmax(logits_b, dim=1)  # (B,H,W)

            # akumulasi IoU global
            tp, fp, fn, tn = smp.metrics.get_stats(
                pred_idx_b, masks_b, mode="multiclass", num_classes=len(IDX2NAME)
            )
            all_tp.append(tp)
            all_fp.append(fp)
            all_fn.append(fn)

            # accuracy global
            correct_pixels += (pred_idx_b == masks_b).sum().item()
            total_pixels   += masks_b.numel()

            # convert ke numpy utk simpan visualisasi
            pred_idx_np = pred_idx_b.cpu().numpy().astype(np.int32)
            true_idx_np = masks_b.cpu().numpy().astype(np.int32)
            images_np   = images_b.cpu().permute(0,2,3,1).numpy()  # (B,H,W,3)

            B = images_np.shape[0]
            for k in range(B):
                sample_id += 1
                img    = images_np[k]
                gt_idx = true_idx_np[k]
                pr_idx = pred_idx_np[k]
                img_vis = np.clip(img * 255, 0, 255).astype(np.uint8)

                persen_k, tingkat_k, penyakit_k = severity_from_pred(pr_idx, gt_idx, use_gt_leaf=True)

                # tulis ke file log
                f_log.write(f"\n=== Sampel {sample_id} ===\n")
                f_log.write(f"Penyakit Terdeteksi      : {penyakit_k}\n")
                f_log.write(f"Persentase Area Penyakit : {persen_k:.2f}%\n")
                f_log.write(f"Tingkat Keparahan        : {tingkat_k}\n")
                f_log.write(f"Unique GT mask values : {np.unique(gt_idx)}\n")
                f_log.write(f"Unique Pred mask values: {np.unique(pr_idx)}\n")

                # simpan gambar
                plt.figure(figsize=(14,4))
                plt.subplot(1,3,1); plt.title(f"Image [{sample_id}]")
                plt.imshow(img_vis); plt.axis("off")
                plt.subplot(1,3,2); plt.title("Ground Truth")
                plt.imshow(decode_segmap(gt_idx)); plt.axis("off")
                plt.subplot(1,3,3); plt.title("Prediction")
                plt.imshow(decode_segmap(pr_idx)); plt.axis("off")

                save_path = os.path.join(hasil_dir, f"visualisasi_sample_{sample_id}.png")
                plt.savefig(save_path, dpi=150)
                plt.close()

    # hitung global metrics
    iou_per_class, miou = compute_global_iou(all_tp, all_fp, all_fn, len(IDX2NAME))
    accuracy = correct_pixels / total_pixels

    # tulis ringkasan global
    f_log.write("\n\n== Evaluation Results (Global) ==\n")
    f_log.write(f"Accuracy : {accuracy:.4f}\n")
    f_log.write(f"mIoU     : {miou:.4f}\n")
    f_log.write("IoU per class:\n")
    for i, iou_c in enumerate(iou_per_class):
        cname = IDX2NAME.get(i, f"Class {i}")
        val = 0.0 if np.isnan(iou_c) else float(iou_c)
        f_log.write(f"  {cname:>16}: {val:.4f}\n")

    f_log.close()
    print(f"\n✅ Semua hasil disimpan di: {hasil_dir}")
    print(f"- Ringkasan & analisis: {log_path}")
    print(f"- Visualisasi per sample: {hasil_dir}/visualisasi_sample_*.png")


# === Contoh pemanggilan ===
epoch = EPOCHS
nama_model = "DeepLabV3plus"
evaluate_and_save(model, test_loader, DEVICE, nama_model=nama_model, epoch=epoch)